# 04. AI Search 검색 및 RAG 테스트

이미 인덱싱된 법률 데이터를 활용하여 다양한 검색 기법을 실습합니다.

| 검색 기법 | 설명 |
|----------|------|
| **키워드 검색** | BM25 전통적 텍스트 매칭 |
| **벡터 검색** | text-embedding-3-large (3072D) 의미 기반 |
| **하이브리드 검색** | 키워드 + 벡터 RRF 결합 |
| **시맨틱 재랭킹** | Azure AI Search Semantic Ranker (L2R) |
| **필터 + 벡터** | 메타데이터 필터링 후 의미 기반 검색 |
| **Multi-Index 검색** | 4개 인덱스 동시 검색 |
| **RAG** | 검색 결과 + GPT-5.4 종합 답변 |
| **Cross-Index RAG** | 여러 법률 소스 통합 RAG |

권장 순서: `02 → 03 → 04`

⚠️ **실행 위치 필수 조건**  
이 노트북은 **내부망(VNet/Private Endpoint)에 연결된 Remote VM**에서 실행해야 합니다.

## 1. 환경 설정

In [2]:
import os, sys
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

from azure.identity import AzureCliCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery, QueryType, QueryCaptionType
from openai import AzureOpenAI

SEARCH_ENDPOINT  = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
OPENAI_ENDPOINT  = os.environ['AZURE_OPENAI_ENDPOINT']
GPT54_DEPLOY     = os.environ.get('AZURE_OPENAI_GPT54_DEPLOYMENT', 'gpt-5.4')
EMBED_DEPLOY     = os.environ.get('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large')

from src.search.legal_indexes import LegalIndexManager, PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX

# Pin tenant — DefaultAzureCredential may pick the wrong VS Code / shared-cache tenant.
AZ_TENANT_ID = os.environ.get('AZURE_TENANT_ID', 'ebed7cce-48ed-45fe-ad61-fddfc8e2fb54')
credential = AzureCliCredential(tenant_id=AZ_TENANT_ID)

mgr = LegalIndexManager(
    endpoint=SEARCH_ENDPOINT,
    admin_key=os.environ.get('AZURE_SEARCH_ADMIN_KEY'),
)

# OpenAI 클라이언트
token_provider = get_bearer_token_provider(
    credential,
    'https://cognitiveservices.azure.com/.default'
)
openai_client = AzureOpenAI(
    azure_endpoint=OPENAI_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version='2025-01-01-preview',
)

print(f"Search Endpoint : {SEARCH_ENDPOINT}")
print(f"OpenAI Endpoint : {OPENAI_ENDPOINT}")
print(f"GPT-5.4 배포명  : {GPT54_DEPLOY}")
print()

# 인덱스 통계
stats = mgr.get_all_stats()
print("인덱스 문서 개수:")
for s in stats:
    print(f"  {s['index_name']}: {s['document_count']:,}건")

Search Endpoint : https://search-ragi-63325wdo.search.windows.net
OpenAI Endpoint : https://ais-ragi-63325wdo.cognitiveservices.azure.com/
GPT-5.4 배포명  : gpt-5.4

인덱스 문서 개수:
  prec-court-index: 20,505건
  const-court-index: 34,106건
  legis-interp-index: 8,715건
  admin-appeal-index: 9,300건


## 2. 인덱스 스키마 (실제 배포 기준)

### 공통 설계 원칙

| 구분 | 타입 | 설정 | 용도 |
|------|------|------|------|
| **Key 필드** | `Edm.String` | `key=true, analyzer=keyword` | 문서 고유 ID |
| **날짜 메타** | `Edm.DateTimeOffset` | `filterable, sortable, facetable, retrievable` | 날짜 범위 필터·정렬 |
| **열거형 메타** | `Edm.String` | `searchable, filterable, sortable, facetable` | 심급·유형 패싯 |
| **다중값 메타** | `Collection(Edm.String)` | `searchable, filterable, facetable` | 관련법령·주제어 (any/all 람다) |
| **본문 텍스트** | `Edm.String` | `searchable, analyzer=ko.microsoft` | 한국어 형태소 분석 |
| **장문 본문** | `Edm.String` | `searchable, analyzer=ko_safe` (filter/sort/facet 모두 false) | 32K 토큰 한도 회피 — `_text_long()` |
| **벡터 임베딩** | `Collection(Edm.Single)` | `dim=3072, vectorSearchProfile=hnsw` | 의미 기반 검색 |

**주요 설계 결정**

- **`ko.microsoft` vs `ko_safe`**: 짧은 메타·요약 필드는 표준 Microsoft 한국어 analyzer 사용. `fullText`/`reason` 같은 장문 필드는 `ko_safe` (CustomAnalyzer = `PatternReplaceCharFilter`로 한자/가나 제거 + 200자 단위 강제 분할 → `MicrosoftLanguageTokenizer(korean)` → `lowercase`)
- **장문 필드의 filter/sort/facet 비활성**: AI Search는 filterable/sortable/facetable 가 켜진 String 필드를 **단일 토큰**으로 색인하기 때문에 32 766 byte 한도를 넘으면 색인 실패. 장문은 검색만 켜고 메타 옵션은 모두 끔
- **임베딩 분리 보호**: `SplitSkill(maximumPageLength=5000)` → 첫 페이지만 `text-embedding-3-large` 호출 → 8K 토큰 한도 회피
- **`Collection(Edm.String)`**: "민법,형법" 단일 텍스트 불가 → 배열로 분리하여 람다 필터 가능
- **임베딩 필드 선택**: 전체 본문이 아닌 *요약/회답/요지* 필드만 임베딩 → 비용·정확도 균형

> **필드 attribute 표기**: `s`=searchable, `f`=filterable, `o`=sortable, `a`=facetable, `r`=retrievable, **KEY**=key

---

### 1. `prec-court-index` — 판례 (대법원·고등·지방법원)
**docs = 18,841 / storage = 522.7 MiB / vector = 108.0 MiB**

| 필드명 | 타입 | Attributes | Analyzer | 비고 |
|--------|------|------------|----------|------|
| `id` | String | KEY,s,f,o,a,r | keyword | 고유 ID |
| `courtName` | String | s,f,o,a,r | — | 법원명 |
| `caseNumber` | String | s,f,o,a,r | — | 사건번호 |
| `judgmentDate` | DateTimeOffset | f,o,a,r | — | 선고일자 |
| `courtLevel` | String | s,f,o,a,r | — | 심급 (1심/2심/3심) |
| `caseType` | String | s,f,o,a,r | — | 사건종류 |
| `status` | String | s,f,o,a,r | — | 진행상태 |
| `relatedLaws` | Collection(String) | s,f,a,r | — | 관련법령 배열 |
| `keywords` | Collection(String) | s,f,a,r | — | 주제어 배열 |
| `registrationDate` | DateTimeOffset | f,o,a,r | — | 등록일자 |
| `caseName` | String | s,r | ko.microsoft | 사건명 |
| `holdings` | String | s,r | ko.microsoft | 판시사항 |
| `summary` | String | s,r | ko.microsoft | 판결요지 ← **임베딩 대상** |
| `fullText` | String | s,r | **ko_safe** | 판결 전문 (장문) |
| `summaryEmbedding` | Collection(Single) | s | — | dim=3072 (HNSW) |

**Semantic Config**: `prec-semantic` (title=`caseName` / content=`holdings,summary` / keywords=`keywords,relatedLaws`)

---

### 2. `const-court-index` — 헌법재판소 결정례
**docs = 24,980 / storage = 338.5 MiB / vector = 44.4 MiB**

| 필드명 | 타입 | Attributes | Analyzer | 비고 |
|--------|------|------------|----------|------|
| `id` | String | KEY,s,f,o,a,r | keyword | 고유 ID |
| `caseNumber` | String | s,f,o,a,r | — | 사건번호 |
| `decisionDate` | DateTimeOffset | f,o,a,r | — | 결정일자 |
| `decisionType` | String | s,f,o,a,r | — | 결정유형 |
| `relatedLaws` | Collection(String) | s,f,a,r | — | 관련법령 배열 |
| `keywords` | Collection(String) | s,f,a,r | — | 주제어 배열 |
| `fiscalYear` | String | s,f,o,a,r | — | 귀속년도 |
| `registrationDate` | DateTimeOffset | f,o,a,r | — | 등록일자 |
| `caseName` | String | s,f,o,a,r | ko.microsoft | 사건명 |
| `holdings` | String | s,f,o,a,r | ko.microsoft | 판시사항 |
| `summary` | String | s,f,o,a,r | ko.microsoft | 결정요지 ← **임베딩 대상** |
| `fullText` | String | s,r | **ko_safe** | 전문 (장문) |
| `summaryEmbedding` | Collection(Single) | s | — | dim=3072 (HNSW) |

**Semantic Config**: `const-semantic` (title=`caseName` / content=`holdings,summary` / keywords=`keywords,relatedLaws`)

---

### 3. `legis-interp-index` — 법제처 해석례
**docs = 8,715 / storage = 410.0 MiB / vector = 102.5 MiB**

| 필드명 | 타입 | Attributes | Analyzer | 비고 |
|--------|------|------------|----------|------|
| `id` | String | KEY,s,f,o,a,r | keyword | 고유 ID |
| `docNumber` | String | s,f,o,a,r | — | 문서번호 |
| `replyDate` | DateTimeOffset | f,o,a,r | — | 회신일자 |
| `interpType` | String | s,f,o,a,r | — | 안건종류 |
| `relatedLaws` | Collection(String) | s,f,a,r | — | 관련법령 배열 |
| `keywords` | Collection(String) | s,f,a,r | — | 주제어 배열 |
| `registrationDate` | DateTimeOffset | f,o,a,r | — | 등록일자 |
| `title` | String | s,r | ko.microsoft | 안건명 |
| `querySummary` | String | s,r | ko.microsoft | 질의요지 |
| `reply` | String | s,r | ko.microsoft | 회답 ← **임베딩 대상** |
| `reason` | String | s,r | **ko_safe** | 이유 (장문) |
| `summaryEmbedding` | Collection(Single) | s | — | dim=3072 (HNSW) |

**Semantic Config**: `interp-semantic` (title=`title` / content=`querySummary,reply` / keywords=`keywords,relatedLaws`)

---

### 4. `admin-appeal-index` — 행정심판 재결례
**docs = 6,830 / storage = 342.6 MiB / vector = 80.3 MiB**

| 필드명 | 타입 | Attributes | Analyzer | 비고 |
|--------|------|------------|----------|------|
| `id` | String | KEY,s,f,o,a,r | keyword | 고유 ID |
| `caseNumber` | String | s,f,o,a,r | — | 사건번호 |
| `decisionDate` | DateTimeOffset | f,o,a,r | — | 재결일자 |
| `decisionType` | String | s,f,o,a,r | — | 재결유형 (인용/기각/각하) |
| `relatedLaws` | Collection(String) | s,f,a,r | — | 관련법령 배열 |
| `keywords` | Collection(String) | s,f,a,r | — | 주제어 배열 |
| `committee` | String | s,f,o,a,r | — | 심판위원회 |
| `registrationDate` | DateTimeOffset | f,o,a,r | — | 등록일자 |
| `caseName` | String | s,f,o,a,r | ko.microsoft | 사건명 |
| `order` | String | s,f,o,a,r | ko.microsoft | 주문 |
| `reasonSummary` | String | s,f,o,a,r | ko.microsoft | 재결요지 ← **임베딩 대상** |
| `fullText` | String | s,r | **ko_safe** | 전문 (장문) |
| `summaryEmbedding` | Collection(Single) | s | — | dim=3072 (HNSW) |

**Semantic Config**: `admin-semantic` (title=`caseName` / content=`order,reasonSummary` / keywords=`keywords,relatedLaws`)

---

### Collection(String) OData 필터 예시

```
# 특정 법령 포함
filter="relatedLaws/any(law: law eq '민법')"

# 여러 법령 중 하나
filter="relatedLaws/any(law: law eq '민법' or law eq '형법')"

# 법령 + 날짜 범위 복합
filter="relatedLaws/any(law: law eq '민법') and judgmentDate ge 2020-01-01T00:00:00Z"

# 심급 + 상태 + 날짜
filter="courtLevel eq '3심' and status eq '확정' and judgmentDate ge 2020-01-01T00:00:00Z"
```

> 4개 인덱스의 인덱싱 시간/비용/스토리지 등 종합 리포트는 [REPORT.md](../REPORT.md) 참고.


## 3. 단일 인덱스 검색 (판례)

### 3-1. 키워드 검색

In [4]:
query = "비리 공무원 징계에 관련된 판례"
results = mgr.search(PREC_INDEX, query=query, top=3, use_semantic=False)
results

# print(f"\n[키워드 검색] '{query}' (판례, 상위 3건)\n")
# for i, r in enumerate(results, 1):
#     print(f"{i}. [{r['caseName']}] {r['caseName'][:80]}")
#     print(f"   법원: {r['courtName']}, 심급: {r['courtLevel']}")
#     print(f"   점수: {r['score']:.4f}")
#     print()

[{'id': '194242',
  'courtName': '대법원',
  'caseNumber': '2016두33339',
  'judgmentDate': '2018-03-13T00:00:00Z',
  'courtLevel': None,
  'caseType': None,
  'status': None,
  'relatedLaws': [],
  'keywords': [],
  'registrationDate': None,
  'caseName': '퇴교처분취소',
  'holdings': '[1] 행정청이 징계와 같은 불이익처분절차에서 징계심의대상자가 선임한 변호사가 징계위원회에 출석하여 징계심의대상자를 위하여 필요한 의견을 진술하는 것을 거부할 수 있는지 여부 (원칙적 소극) [2] 행정절차법의 적용이 제외되는 공무원 인사관계 법령에 의한 처분에 관한 사항의 의미 및 이러한 법리가 육군3사관학교 생도에 대한 퇴학처분에도 적용되는지 여부 (적극) / 생도에 대한 퇴학처분과 같이 신분을 박탈하는 징계처분이 행정절차법의 적용이 제외되는 경우인 행정절차법 시행령 제2조 제8호 에 해당하는지 여부 (소극) [3] 육군3사관학교 사관생도에 대한 징계절차에서 징계심의대상자가 대리인으로 선임한 변호사가 징계위원회 심의에 출석하여 진술하는 것을 막은 경우, 징계처분이 위법하여 취소되어야 하는지 여부 (원칙적 적극) 및 징계처분이 취소되지 않는 예외적인 경우',
  'summary': '[1] 행정절차법 제12조 제1항 제3호 , 제2항 , 제11조 제4항 본문에 따르면, 당사자 등은 변호사를 대리인으로 선임할 수 있고, 대리인으로 선임된 변호사는 당사자 등을 위하여 행정절차에 관한 모든 행위를 할 수 있다고 규정되어 있다. 위와 같은 행정절차법령의 규정과 취지, 헌법상 법치국가원리와 적법절차원칙에 비추어 징계와 같은 불이익처분절차에서 징계심의대상자에게 변호사를 통한 방어권의 행사를 보장하는 것이 필요하고, 징계심의대상자가 선임한 변호사가 징계위원회에 출석하여 징계심의대상자

### 3-2. 하이브리드 검색 (키워드 + 벡터)

In [ ]:
query = "공직 선거 및 선거 부정방지 관련 법령 및 판례 조사해줘 "

# 쿼리 임베딩 생성
embedding_response = openai_client.embeddings.create(
    input=query,
    model=EMBED_DEPLOY,
)
embedding = embedding_response.data[0].embedding

results = mgr.search(PREC_INDEX, query=query, embedding=embedding, top=3, use_semantic=True)
results

# print(f"\n[하이브리드 + 시맨틱 재랭킹] '{query}' (상위 3건)\n")
# for i, r in enumerate(results, 1):
#     print(f"{i}. [{r['caseNumber']}] {r['caseName'][:80]}")
#     print(f"   점수: {r['score']:.4f}")
#     print(f"   판결요지: {r.get('summary', '')[:120]}...")
#     print()

## 4. 필터 기반 검색 (메타데이터 필터링)

Collection(String) 필터를 사용한 정밀 검색 예시

In [ ]:
# 대법원 세무 사건만 필터
print("\n[필터 검색] 판례 중 대법원(3심) 세무 사건\n")
results = mgr.search(
    PREC_INDEX,
    query="*",
    filter_expr="courtLevel eq '3심'",
    select=["caseName", "caseNumber", "courtName", "courtLevel"],
    top=5,
)
for r in results[:3]:
    print(f"[{r['caseNumber']}] {r['caseName'][:80]}")
    print(f"  {r['courtName']} - {r['courtLevel']}\n")

## 5. Multi-Index 통합 검색 (4개 인덱스 동시 검색)

4개 법률 인덱스에서 동시에 검색하여 점수 기반으로 통합 정렬합니다.

In [ ]:
def cross_index_search(query: str, top_per_index: int = 3) -> list:
    """4개 인덱스 통합 검색"""
    # 임베딩 생성
    embedding_response = openai_client.embeddings.create(
        input=query,
        model=EMBED_DEPLOY,
    )
    embedding = embedding_response.data[0].embedding
    
    # 각 인덱스별 검색
    INDEX_LABELS = {
        PREC_INDEX: "판례",
        CONST_INDEX: "헌재결정례",
        INTERP_INDEX: "법제처해석례",
        ADMIN_INDEX: "행정심판재결례",
    }
    
    all_results = []
    for idx_name in [PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX]:
        results = mgr.search(idx_name, query=query, embedding=embedding, top=top_per_index)
        for r in results:
            r["_index"] = idx_name
            r["_label"] = INDEX_LABELS[idx_name]
        all_results.extend(results)
    
    # 점수 기반 정렬
    all_results.sort(key=lambda x: x.get("score", 0), reverse=True)
    return all_results


# 테스트
query = "개인정보 보호와 관련된 법률적 쟁점"
results = cross_index_search(query, top_per_index=2)

print(f"\n[Multi-Index 검색] '{query}'\n")
print(f"총 {len(results)}건 (4개 인덱스 통합, 상위 8건 표시)\n")

for i, r in enumerate(results[:8], 1):
    title = r.get("caseName") or r.get("title", "N/A")
    print(f"{i}. [{r['_label']}] (score: {r['score']:.4f})")
    print(f"   {title[:80]}")
    print()

## 6. RAG — GPT-5.4 기반 질의응답 (판례)

In [ ]:
def rag_single_index(question: str, index_name: str, top_k: int = 3) -> str:
    """단일 인덱스 RAG"""
    # 임베딩 생성
    embedding_response = openai_client.embeddings.create(
        input=question,
        model=EMBED_DEPLOY,
    )
    embedding = embedding_response.data[0].embedding
    
    # 검색
    results = mgr.search(index_name, query=question, embedding=embedding, top=top_k)
    
    # 컨텍스트 구성
    context_parts = []
    for r in results:
        title = r.get("caseName") or r.get("title", "")
        case_no = r.get("caseNumber") or r.get("docNumber", "")
        body = "\n".join([
            r.get(f, "") for f in ["holdings", "summary", "querySummary", "reply", "order", "reasonSummary"]
            if r.get(f)
        ])
        context_parts.append(f"[{title}] ({case_no})\n{body}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # GPT-5.4로 응답 생성
    response = openai_client.chat.completions.create(
        model=GPT54_DEPLOY,
        messages=[
            {
                'role': 'system',
                'content': '당신은 한국 법률 전문가입니다. 제공된 판례 자료를 바탕으로 사용자 질문에 정확하고 체계적으로 답변하세요. 근거 판례번호를 명시하세요.',
            },
            {
                'role': 'user',
                'content': f'## 참고 판례\n\n{context}\n\n## 질문\n\n{question}',
            },
        ],
        max_completion_tokens=1000,
    )
    
    return response.choices[0].message.content


# 테스트
question = "대법원 판례 기준으로 개인정보 제3자 제공의 위법성 판단 요소와 손해배상 인정 요건을 사건번호와 함께 설명해 주세요."
answer = rag_single_index(question, PREC_INDEX, top_k=3)

print(f"\n[RAG] 질문: {question}\n")
print("=" * 80)
print(answer)
print("=" * 80)


## 7. Cross-Index RAG — 여러 법률 소스 통합 답변

In [ ]:
def cross_index_rag(question: str, top_per_index: int = 2) -> str:
    """Cross-Index RAG: 4개 인덱스 검색 → GPT-5.4 응답"""
    results = cross_index_search(question, top_per_index=top_per_index)
    
    context_parts = []
    for r in results[:8]:  # 상위 8건
        label = r.get("_label", "기타")
        title = r.get("caseName") or r.get("title", "")
        case_no = r.get("caseNumber") or r.get("docNumber", "")
        body = "\n".join([
            r.get(f, "") for f in ["holdings", "summary", "querySummary", "reply", "order", "reasonSummary"]
            if r.get(f)
        ])
        context_parts.append(f"[{label}] {title} ({case_no})\n{body}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # GPT-5.4로 응답 생성
    response = openai_client.chat.completions.create(
        model=GPT54_DEPLOY,
        messages=[
            {
                'role': 'system',
                'content': (
                    '당신은 한국 법률 전문가입니다. '
                    '판례, 헌법재판소 결정례, 법제처 해석례, 행정심판 재결례 등 '
                    '다양한 법률 자료를 종합하여 사용자 질문에 정확하고 체계적으로 답변하세요.\\n'
                    '근거가 되는 사건번호와 자료 종류를 명시하세요.'
                ),
            },
            {
                'role': 'user',
                'content': f'## 검색된 법률 자료\n\n{context}\n\n## 질문\n\n{question}',
            },
        ],
        max_completion_tokens=1500,
    )
    
    return response.choices[0].message.content


# 테스트
question = "판례·헌법재판소 결정례·법제처 해석례·행정심판 재결례를 비교해, 개인정보 처리 동의 없는 정보 제공이 문제된 대표 사례와 판단 기준을 정리해 주세요."
answer = cross_index_rag(question)

print(f"\n[Cross-Index RAG] 질문: {question}\n")
print("=" * 80)
print(answer)
print("=" * 80)


## 8. Foundry IQ — Agentic (Intelligent) Retrieval

**Foundry IQ** = Microsoft Foundry 의 지능형 검색. 내부적으로 Azure AI Search 의 두 리소스를 사용합니다:

| 리소스 | 역할 |
|---|---|
| **Knowledge Source** | 검색 대상(인덱스/Blob) 한 개를 wrapper. 인덱스당 1개 |
| **Knowledge Agent** | 다중 Knowledge Source 위에서 **질의 계획(query planning)** 을 담당 — LLM(`gpt-4o`/`gpt-4.1` 계열) |

### 동작 방식 — 단일 retrieve 호출

```
[user message + 대화 이력]
        │
        ▼
[Agent: LLM이 질문을 sub-query N개로 분해]   ← maxSubQueries 제한
        │
        ├──► Source 1 (hybrid + semantic rerank, parallel)
        ├──► Source 2  
        └──► Source 3
        │
        ▼
[reranker threshold 통과한 결과만 합쳐서 응답]
        │
        ▼
{ response, references, activity(검색 trace) }
```

### Agentic Retrieval

| 패턴 | 설명 | 작동 |
|---|---|---|
| **A. 단일 호출, 복합 질문** | "X사건과 그 사건이 인용한 선행 판례까지 알려줘" → agent 가 _사전에_ 여러 sub-query 계획 | ✅ |
| **B. Multi-turn 인용 추적** | 1턴: "X사건 알려줘" → 응답에 "참조: 판례 2019도1234" 포함 → 2턴: 이전 응답을 `messages` 에 넣고 "더 알려줘" → agent 가 컨텍스트의 인용번호를 보고 새 sub-query 계획 | ✅ |
| **C. Agent 가 결과를 보고 _자동으로_ 추가 호출** | 현재 API 는 단일 호출 내에서 iterative chase 를 지원하지 **않음** | ❌ |

→ **C 패턴**이 필요하면 클라이언트(예: GPT‑5.4)가 결과의 references 를 파싱해 새 retrieve 를 명시적으로 호출하는 **루프**를 직접 구현해야 합니다 (Agent‑on‑Agent 패턴, 8‑3 셀에서 시연).

> **모델 제약**: Knowledge Agent 는 `gpt-4o, gpt-4o-mini, gpt-4.1, gpt-4.1-mini, gpt-4.1-nano` 만 지원. 본 Lab 은 배포된 `gpt-4o` 사용. 응답 합성용 LLM 은 별도(`gpt-5.4`).


In [ ]:
"""8-1. Knowledge Sources + Knowledge Agent 확인"""
import json, requests

API_VERSION    = "2025-08-01-preview"
PLANNER_DEPLOY = "gpt-4o"
AGENT_NAME     = "legal-knowledge-agent"
AOAI_RESOURCE_URI = "https://ais-ragi-63325wdo.openai.azure.com"

# 각 KS 의 description 은 planner 라우팅 판단에 쓰이므로 corpus 범위 + 키워드 패턴을 풍부하게 작성한다.
INDEX_TO_SRC = {
    PREC_INDEX: (
        "ks-prec",
        ("대한민국 **법원 판례** corpus. 대법원·고등법원·지방법원의 판결문(민사·형사·행정 등) 만 포함한다. "
         "포함 키워드 예: '대법원', '법원 판결', '판례', '○○도○○가합○○○○', '판시사항', '판결요지'. "
         "헌법재판소 결정, 법제처 해석, 행정심판 재결은 이 source 에 없다."),
        "caseName,caseNumber,courtName,courtLevel,judgmentDate,relatedLaws,keywords,holdings,summary",
    ),
    CONST_INDEX: (
        "ks-const",
        ("대한민국 **헌법재판소 결정례** corpus. 헌법재판소가 내린 위헌·합헌·헌법불합치·각하·기각 결정문만 포함한다. "
         "포함 키워드 예: '헌법재판소', '헌재', '위헌', '헌법불합치', '○○헌가○', '○○헌마○', '○○헌바○'. "
         "일반 법원 판례, 법제처 해석, 행정심판 재결은 이 source 에 없다."),
        "caseName,caseNumber,decisionDate,decisionType,relatedLaws,keywords,holdings,summary",
    ),
    INTERP_INDEX: (
        "ks-interp",
        ("대한민국 **법제처 법령해석례** corpus. 법제처가 중앙·지방행정기관 질의에 회답한 법령 해석 사례만 포함한다. "
         "포함 키워드 예: '법제처', '법령해석', '해석례', '회답', '안건번호', '질의요지', '회답요지'. "
         "법원 판례, 헌법재판소 결정, 행정심판 재결은 이 source 에 없다."),
        "title,docNumber,replyDate,interpType,relatedLaws,keywords,querySummary,reply",
    ),
    ADMIN_INDEX: (
        "ks-admin",
        ("대한민국 **행정심판 재결례** corpus. 중앙·시도 행정심판위원회의 재결문(인용·기각·각하) 만 포함한다. "
         "포함 키워드 예: '행정심판', '재결', '행정심판위원회', '○○행심○○○○', '청구취지', '재결주문'. "
         "법원 판례, 헌법재판소 결정, 법제처 해석은 이 source 에 없다."),
        "caseName,caseNumber,decisionDate,decisionType,committee,relatedLaws,keywords,order,reasonSummary",
    ),
}

cred = credential
def _hdr():
    tok = cred.get_token("https://search.azure.com/.default").token
    return {"Authorization": f"Bearer {tok}", "Content-Type": "application/json"}

# KS description 을 갱신하기 위해 항상 PUT (idempotent upsert)
for idx_name, (src_name, desc, fields) in INDEX_TO_SRC.items():
    body = {"name": src_name, "kind": "searchIndex", "description": desc,
            "searchIndexParameters": {"searchIndexName": idx_name, "sourceDataSelect": fields}}
    url = f"{SEARCH_ENDPOINT}/knowledgesources('{src_name}')?api-version={API_VERSION}"
    r = requests.put(url, headers=_hdr(), json=body)
    print(f"  KS '{src_name}' <- {idx_name}: {r.status_code}")

agent_url = f"{SEARCH_ENDPOINT}/agents('{AGENT_NAME}')?api-version={API_VERSION}"
agent_body = {
    "name": AGENT_NAME,
    "description": "Korean legal corpus router",
    "models": [{
        "kind": "azureOpenAI",
        "azureOpenAIParameters": {
            "resourceUri": AOAI_RESOURCE_URI,
            "deploymentId": PLANNER_DEPLOY,
            "modelName": PLANNER_DEPLOY,
        }
    }],
    "knowledgeSources": [
        {"name": s, "alwaysQuerySource": False, "includeReferences": True,
         "includeReferenceSourceData": True, "maxSubQueries": 2, "rerankerThreshold": 1.5}
        for _, (s, _, _) in INDEX_TO_SRC.items()
    ],
    "outputConfiguration": {"modality": "answerSynthesis", "includeActivity": True, "attemptFastPath": False},
    "requestLimits": {"maxRuntimeInSeconds": 60, "maxOutputSize": 16000},
    "retrievalInstructions": (
        "너는 한국 법률 질의 라우터다. 4 개의 knowledge source 가 있으며 각각 서로 다른 corpus 만 담는다:\n"
        "  - ks-prec   = 법원 판례 (대법원·고등·지방법원 판결)\n"
        "  - ks-const  = 헌법재판소 결정례 (위헌·합헌·헌법불합치)\n"
        "  - ks-interp = 법제처 법령해석례 (행정 해석·회답)\n"
        "  - ks-admin  = 행정심판 재결례 (행정심판위원회 재결)\n"
        "\n"
        "## 라우팅 규칙 (반드시 준수)\n"
        "1. **하나의 sub-query 는 정확히 하나의 source 에만 보낸다.** 동일 sub-query 를 여러 source 에 cross-product 로 복제 금지.\n"
        "2. 사용자가 corpus 를 명시했으면 그 source 만 호출한다 (예: '헌법재판소 결정례' → ks-const 만).\n"
        "3. 사용자가 corpus 를 명시하지 않은 경우, sub-query 를 만들 때 **각 sub-query 자체에 어느 corpus 를 찾을지 키워드를 박아 넣고** 그 키워드에 맞는 source 한 곳에만 보낸다. 예: sub-query='임대차 분쟁 대법원 판례' → ks-prec 한 곳.\n"
        "4. sub-query 는 **최대 2 개까지만** 만든다. 모호한 단일 질문은 가장 핵심적인 corpus 1~2 곳만 골라라 (보통 ks-prec 우선).\n"
        "5. 동일 source 에 의미가 거의 같은 sub-query 2 개를 보내지 마라.\n"
        "\n"
        "응답에는 사용한 source 와 사건번호를 명시하라."
    ),
}
r = requests.put(agent_url, headers=_hdr(), json=agent_body)
print(f"✓ Agent '{AGENT_NAME}' 갱신: {r.status_code}")
if r.status_code >= 400:
    print(r.text[:500])


In [ ]:
"""8-2. retrieve helper + 단일 호출 복합 질문

복합 질문을 던져 agent 가 자동으로 여러 sub-query 를 계획하는지 activity 로 검증.
"""
from collections import defaultdict


def retrieve(messages: list, source_filters: dict | None = None) -> dict:
    body = {"messages": messages}
    if source_filters:
        body["knowledgeSourceParams"] = [
            {"kind": "searchIndex", "knowledgeSourceName": k, "filterAddOn": v}
            for k, v in source_filters.items()
        ]
    url = f"{SEARCH_ENDPOINT}/agents('{AGENT_NAME}')/retrieve?api-version={API_VERSION}"
    r = requests.post(url, headers=_hdr(), json=body, timeout=120)
    if r.status_code >= 400:
        print(f"HTTP {r.status_code}\n{r.text[:1500]}")
        r.raise_for_status()
    return r.json()


def show_activity(resp: dict, max_refs: int = 5):
    print("\n--- ACTIVITY (query plan) ---")
    sub_q = []
    for a in resp.get("activity", []):
        t = a["type"]
        if t == "modelQueryPlanning":
            print(f"  [{a['id']}] planning  : in={a.get('inputTokens')}  out={a.get('outputTokens')}  {a.get('elapsedMs')}ms")
        elif t == "modelAnswerSynthesis":
            print(f"  [{a['id']}] synthesis : in={a.get('inputTokens')}  out={a.get('outputTokens')}  {a.get('elapsedMs')}ms")
        elif t == "searchIndex":
            args = a.get("searchIndexArguments", {})
            q = args.get("search", "")
            f = args.get("filter") or ""
            print(f"  [{a['id']}] search    : ks={a['knowledgeSourceName']:<10} hits={a['count']:>3}  q={q[:70]!r}{(' filter='+f) if f else ''}")
            sub_q.append((a["knowledgeSourceName"], q))
        elif t == "semanticReranker":
            print(f"  [{a['id']}] rerank    : in={a.get('inputTokens')}  {a.get('elapsedMs')}ms")
    print(f"\n  -> 총 sub-query: {len(sub_q)}개")

    refs = resp.get("references", [])
    print(f"\n--- REFERENCES (top {min(max_refs, len(refs))}/{len(refs)}) ---")
    for ref in refs[:max_refs]:
        sd = ref.get("sourceData", {})
        title = sd.get("caseName") or sd.get("title", "")
        case = sd.get("caseNumber") or sd.get("docNumber", "")
        print(f"  [{ref.get('rerankerScore', 0):.2f}] {case}  {title[:60]}")


def diagnose_query_plan(resp: dict):
    """query-plan 중복/전수조회 패턴을 간단 진단"""
    searches = [a for a in resp.get("activity", []) if a.get("type") == "searchIndex"]
    if not searches:
        print("\n--- PLAN DIAGNOSIS ---")
        print("searchIndex activity 없음")
        return

    by_query = defaultdict(set)
    for a in searches:
        q = (a.get("searchIndexArguments", {}).get("search") or "").strip()
        ks = a.get("knowledgeSourceName", "")
        by_query[q].add(ks)

    duplicated = {q: ks_set for q, ks_set in by_query.items() if len(ks_set) > 1}
    total = len(searches)
    unique_pairs = len({(a.get("knowledgeSourceName"), (a.get("searchIndexArguments", {}).get("search") or "").strip()) for a in searches})

    print("\n--- PLAN DIAGNOSIS ---")
    print(f"searchIndex 호출 수: {total}")
    print(f"(source, query) 유니크 쌍: {unique_pairs}")
    print(f"여러 source에 반복된 query 수: {len(duplicated)}")
    for q, ks_set in list(duplicated.items())[:5]:
        print(f"  - {q[:80]!r} -> {sorted(ks_set)}")



def show_response(resp: dict):
    print("\n--- AGENT RESPONSE (synthesized answer) ---")
    for msg in resp.get("response", []):
        for c in msg.get("content", []):
            if c.get("type") == "text":
                print(c["text"][:1500])


# ===== 시나리오 A: 단일 호출, 모호한 질문 =====
question_A = "개인정보 제3자 제공 분쟁에서 참고해야 할 대표 법률 사례와 판단 기준을 정리해 주세요."
print(f"### 시나리오 A — 단일 호출 (모호한 질문 라우팅 테스트)")
print(f"질문: {question_A}\n")

resp_A = retrieve([{"role": "user", "content": [{"type": "text", "text": question_A}]}])
show_activity(resp_A)
diagnose_query_plan(resp_A)
show_response(resp_A)


In [ ]:
"""8-3. 시나리오 B — Multi-turn 인용 추적

이전 turn 의 agent 응답을 conversation history 로 다음 retrieve 호출에 포함하면,
agent 의 query planner 는 응답에 언급된 사건번호/법령을 보고 새 sub-query 를 자동 계획.
"""
import re

turn1_q = "임대차 보증금 반환 관련 분쟁에 대해 판례 · 헌법재판소 결정례 · 법제처 해석례 · 행정심판 재결례를 각각 1건 이상 소개하고 사건번호를 함께 알려주세요."
print(f"### 시나리오 B-Turn 1\n질문: {turn1_q}\n")

resp_B1 = retrieve([{"role": "user", "content": [{"type": "text", "text": turn1_q}]}])
show_activity(resp_B1, max_refs=3)

turn1_answer = ""
for msg in resp_B1.get("response", []):
    for c in msg.get("content", []):
        if c.get("type") == "text":
            turn1_answer = c["text"]
print(f"\n--- Turn 1 응답 (앞 600자) ---\n{turn1_answer[:600]}\n...")

cited = re.findall(r"\d{4}[가-힣]{1,2}\d{2,6}", turn1_answer)
print(f"\n응답에서 추출한 사건번호: {sorted(set(cited))[:5]}")

turn2_q = "위에서 언급된 사례 중 핵심 1-2건을 골라 사실관계와 판단이유를 자세히 설명해 주세요."
messages_B2 = [
    {"role": "user",      "content": [{"type": "text", "text": turn1_q}]},
    {"role": "assistant", "content": [{"type": "text", "text": turn1_answer}]},
    {"role": "user",      "content": [{"type": "text", "text": turn2_q}]},
]
print(f"\n### 시나리오 B-Turn 2\n질문: {turn2_q}\n")
resp_B2 = retrieve(messages_B2)
show_activity(resp_B2, max_refs=5)

turn2_queries = [
    a["searchIndexArguments"]["search"]
    for a in resp_B2.get("activity", [])
    if a["type"] == "searchIndex"
]
hits = [q for q in turn2_queries if any(c in q for c in set(cited))]
print(f"\n→ Turn 2 sub-query 중 turn 1 인용번호 포함: {len(hits)}/{len(turn2_queries)}")
for q in hits[:3]:
    print(f"   • {q}")

In [ ]:
"""8-4. 시나리오 C — Agent-on-Agent 자동 인용 추적 루프

Knowledge Agent 자체는 단일 호출 안에서 결과를 보고 추가 호출하지 않음.
"자동으로 인용 문서를 따라가는" 동작을 만들려면 클라이언트 루프 필요:
  1. 첫 retrieve → references 의 sourceData 에서 relatedLaws / 사건번호 추출
  2. 그 키워드로 새 retrieve 호출
  3. 두 결과 합쳐 최종 답변 합성

이 셀은 그 패턴을 시연 (max 2 hop).
"""
def extract_followup_terms(refs: list, top_n: int = 5) -> list[str]:
    """references 의 relatedLaws / keywords 에서 후속 검색용 키워드 추출"""
    terms: dict[str, int] = {}
    for ref in refs:
        sd = ref.get("sourceData", {})
        for fld in ("relatedLaws", "keywords"):
            val = sd.get(fld)
            if isinstance(val, list):
                for v in val:
                    if v and 2 <= len(v) <= 30:
                        terms[v] = terms.get(v, 0) + 1
        # 사건번호도 후속 단서
        cn = sd.get("caseNumber") or sd.get("docNumber")
        if cn:
            terms[cn] = terms.get(cn, 0) + 1
    # 빈도 상위
    return [k for k, _ in sorted(terms.items(), key=lambda x: -x[1])[:top_n]]


def agentic_chase(initial_q: str, hops: int = 2) -> dict:
    history = [{"role": "user", "content": [{"type": "text", "text": initial_q}]}]
    all_refs: list = []
    hop_log = []

    for hop in range(hops):
        print(f"\n=== HOP {hop+1} ===")
        resp = retrieve(history)
        show_activity(resp, max_refs=3)

        # 응답 누적
        for msg in resp.get("response", []):
            history.append(msg)
        all_refs.extend(resp.get("references", []))
        hop_log.append({
            "hop": hop+1,
            "subqueries": [a.get("searchIndexArguments", {}).get("search")
                           for a in resp.get("activity", []) if a["type"] == "searchIndex"],
        })

        if hop == hops - 1:
            break
        # 다음 hop 키워드 추출 후 새 user message 추가
        followups = extract_followup_terms(resp.get("references", []), top_n=5)
        print(f"\n→ HOP {hop+1} references 에서 추출한 후속 키워드: {followups}")
        if not followups:
            print("  (후속 키워드 없음 — 루프 종료)")
            break
        next_q = (
            f"앞서 찾은 자료들에서 다음 키워드/법령/사건번호와 관련된 추가 자료를 찾아 더 자세히 설명해주세요: "
            + ", ".join(followups)
        )
        history.append({"role": "user", "content": [{"type": "text", "text": next_q}]})

    return {"history": history, "references": all_refs, "hops": hop_log}


# 실행
print("### 시나리오 C — 클라이언트 루프 자동 인용 추적")
result = agentic_chase("개인정보 제3자 제공 관련 분쟁에서 판례·헌재·해석례·행심 재결례를 함께 찾아 핵심 판단 기준을 비교해 주세요", hops=2)

print("\n\n=== 최종 요약 ===")
for h in result["hops"]:
    print(f"HOP {h['hop']}: {len(h['subqueries'])}개 sub-query")
    for q in h["subqueries"][:5]:
        print(f"   • {(q or '')[:80]}")
print(f"누적 references: {len(result['references'])}건")


### 8-5. 시나리오 D — LLM-as-Judge 자동 루프 (클라이언트 측)

8-7 (시나리오 F) 과 **동일한 chase 질문 + 동일한 retrieval 진단** 을 사용해 세 패턴을 직접 비교한다.

| 시나리오 | 루프 위치 | tool wrapper | system prompt | 기대 동작 |
|---|---|---|---|---|
| **D (이 셀)** | 클라이언트 (chat.completions 루프) | 간단한 retrieve + 중복 차단 | 일반 리서치 지시 | LLM 이 같은 query 를 반복하지 않고 인용 사건번호를 chase 하는가? |
| **E (8-6)** | Foundry Agent Service (서버 루프) | 동일 wrapper | 일반 리서치 지시 | 서버 자율 루프가 동일 chase 행동을 보이는가? |
| **F (8-7)** | Foundry Agent Service | 동일 wrapper | **strict** chase 규칙 (사전지식 금지·중복 금지 명시) | strict 지시가 진단 지표를 얼마나 개선하는가? |

→ D/E 에서는 instructions 를 일부러 단순하게 두고, F 의 strict prompt 와 결과 차이(첫 호출 source 선택, grounded 비율, 중복 차단 횟수) 를 비교하는 것이 목적이다.

**클라이언트 루프 핵심:** `openai_client.chat.completions.create(tools=[...])` 를 우리가 직접 while 루프로 돌린다. 서버는 한 turn 만 plan 하고 `tool_calls` 를 돌려줄 뿐, 다음 turn 호출 책임은 클라이언트.


In [ ]:
"""8-5. 시나리오 D — LLM-as-Judge 자동 tool 루프 (자기-완결 셀)

핵심: openai_client.chat.completions.create(tools=[...]) 를 우리(클라이언트)가 while 로 돌린다.
서버는 한 turn 만 plan 하고 tool_calls 를 돌려줄 뿐, 다음 turn 호출 책임은 우리에게 있음.
"""
import json, time
from requests.exceptions import HTTPError

PLANNER_DEPLOY_D = "gpt-4o"   # tool 호출 가능한 모델
COMPLEX_QUESTION = "다리가 무너져서 사람들이 많이 죽었는데, 관련된 사람들이 다 같이 처벌받은 판례 있어? 그리고 그 판례에서 인용한 판례도 같이 찾아줘."


# ── tool 본체: Knowledge Agent retrieve 호출 (429 backoff 포함) ──
def tool_knowledge_agent_retrieve(query: str, source_filter: str = "all") -> str:
    """[Tool] 한국 법률 자료 4종(ks-prec/ks-const/ks-interp/ks-admin)에서 자연어 질의로 검색·요약."""
    body: dict = {"messages": [{"role": "user", "content": [{"type": "text", "text": query}]}]}
    if source_filter and source_filter != "all":
        body["knowledgeSourceParams"] = [
            {"kind": "searchIndex", "knowledgeSourceName": source_filter, "filterAddOn": ""}
        ]
    url = f"{SEARCH_ENDPOINT}/agents('{AGENT_NAME}')/retrieve?api-version={API_VERSION}"
    delay = 5
    for attempt in range(4):
        r = requests.post(url, headers=_hdr(), json=body, timeout=120)
        if r.status_code == 429:
            print(f"   ⏳ 429 — {delay}s 대기 ({attempt+1}/4)")
            time.sleep(delay); delay *= 2
            continue
        r.raise_for_status()
        resp = r.json()
        break
    else:
        raise RuntimeError("retrieve 429 retries exhausted")

    answer = "\n".join(
        c.get("text", "") for m in resp.get("response", []) for c in m.get("content", [])
        if c.get("type") == "text"
    ).strip() or "(no answer)"
    refs = []
    for ref in resp.get("references", [])[:8]:
        sd = ref.get("sourceData", {})
        title = sd.get("caseName") or sd.get("title") or sd.get("docNumber") or "?"
        case_no = sd.get("caseNumber") or sd.get("docNumber", "")
        refs.append(f"- [{case_no}] {title}")
    sub_q = sum(1 for a in resp.get("activity", []) if a.get("type") == "searchIndex")
    return json.dumps({"answer": answer[:2000], "references": refs, "sub_queries_executed": sub_q},
                      ensure_ascii=False)


# ── tool schema (function calling) ──
TOOL_DEFS = [{
    "type": "function",
    "function": {
        "name": "tool_knowledge_agent_retrieve",
        "description": "한국 법률 자료(판례/헌재/법제처/행심) 통합 검색. source_filter 로 특정 소스만 호출 가능.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "자연어 검색 쿼리"},
                "source_filter": {
                    "type": "string",
                    "enum": ["all", "ks-prec", "ks-const", "ks-interp", "ks-admin"],
                    "description": "특정 소스 한정 (기본 'all')",
                },
            },
            "required": ["query"],
        },
    },
}]


# ── ★ 핵심: 클라이언트 측 while 루프 ★ ──
def autonomous_agent(question: str, max_iters: int = 8) -> dict:
    messages = [
        {"role": "system", "content": (
            "너는 한국 법률 리서치 어시스턴트다. tool_knowledge_agent_retrieve 를 자유롭게 호출해 "
            "자료를 모은 뒤, 충분하다고 판단되면 사건번호를 인용한 종합 답변을 한국어로 생성하라.\n"
            "검색 결과의 answer 나 references 에서 다른 사건번호가 인용되어 있으면, "
            "그 사건번호로 추가 검색해 인용 판례의 내용도 함께 조사하라.\n"
            "필요한 경우 여러 source(ks-prec, ks-const, ks-interp, ks-admin) 를 호출해 자료를 보강하되, "
            "질문에 무관한 source 까지 강제로 호출할 필요는 없다."
        )},
        {"role": "user", "content": question},
    ]
    tool_calls_total = 0
    for it in range(max_iters):
        resp = openai_client.chat.completions.create(
            model=PLANNER_DEPLOY_D,
            messages=messages,
            tools=TOOL_DEFS,
            tool_choice="auto",
        )
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            print(f"\n🏁 iter={it+1} finish=stop  (총 tool 호출 {tool_calls_total}회)")
            break
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            print(f"  [iter {it+1}] tool: {tc.function.name}  source={args.get('source_filter','all')}  q={args.get('query','')[:60]!r}")
            try:
                result = tool_knowledge_agent_retrieve(**args)
            except Exception as e:
                result = json.dumps({"error": str(e)}, ensure_ascii=False)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
            tool_calls_total += 1
    print("\n=== 최종 답변 (D — 클라이언트 루프) ===")
    print(messages[-1].get("content", "")[:2000])
    return {"messages": messages, "tool_calls": tool_calls_total}


print("### 시나리오 D — LLM-as-Judge 자동 tool 루프\n")
print(f"질문:\n{COMPLEX_QUESTION}\n")
result_D = autonomous_agent(COMPLEX_QUESTION)

### 8-6. 시나리오 E — Foundry Agent Service (서버 측 자율 루프)

시나리오 D 와 **완전히 동일한 chase 질문 + 동일한 진단 지표** 를 사용한다. 차이는 LLM 자체-판단 루프가 어디서 도는가:

| 항목 | Scenario D (Chat Completions + tools) | Scenario E (Foundry Agent Service) |
|---|---|---|
| Thread/대화 상태 저장 | 클라이언트 messages 리스트 | Azure 서버 측 |
| Tool 등록 위치 | 매 요청 `tools=[...]` | Agent 정의에 영구 등록 |
| Built-in tools | 없음 (직접 wrap) | **`AzureAISearchTool`, `FileSearchTool`, `CodeInterpreter`, `BingGroundingTool`** 등 즉시 사용 |
| 모니터링/Trace | 직접 구현 | Foundry Portal 에 자동 기록 |
| Multi-agent 오케스트레이션 | 직접 | `connected_agents` 로 agent-to-agent |

E 는 일반 리서치 지시(strict chase 규칙 없음) 만 주어 **Foundry 서버 루프 자체의 자연스러운 chase 능력** 을 본다. F (8-7) 는 같은 인프라에 strict 규칙을 추가했을 때 차이를 본다.

**선결 인프라 (한 번만 수행):**

1. **AIServices 계정에 Foundry project 활성화** — `infra/<region>/main.bicep` 의 `openai` 모듈이 자동 처리 (`allowProjectManagement=true` + `proj-ragi-default`). 노트북 01 을 끝까지 실행했다면 적용됨.

2. **사용자에게 project RBAC 부여** (`Azure AI User`):
   ```bash
   PROJECT_ID=$(az cognitiveservices account project show \
     --account-name "$AOAI_ACCOUNT" -g "$RESOURCE_GROUP" -n proj-ragi-default \
     --query id -o tsv)
   az role assignment create --assignee <user-object-id> \
     --role "Azure AI User" --scope "$PROJECT_ID"
   ```

3. **SDK 설치** (이미 `pyproject.toml` 에 추가됨): `uv pip install -e .`

**환경변수:** `.env` 에 `FOUNDRY_PROJECT_ENDPOINT=https://<aoai-account>.services.ai.azure.com/api/projects/proj-ragi-default` 추가 (노트북 01 의 5d 단계에서 자동 작성됨).


In [ ]:
"""8-6. 시나리오 E — Foundry Agent Service: 서버 측 자율 루프"""
import os, json, time, requests
from dotenv import load_dotenv

load_dotenv(override=True)

SEARCH_ENDPOINT = os.environ.get("SEARCH_ENDPOINT") or os.environ.get("AZURE_SEARCH_ENDPOINT") or os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AGENT_NAME      = os.getenv("KNOWLEDGE_AGENT_NAME", "legal-knowledge-agent")
API_VERSION     = "2025-08-01-preview"
FOUNDRY_PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]

cred = credential  # reuse pinned-tenant credential from setup cell
_search_token = cred.get_token("https://search.azure.com/.default").token


def _retrieve(messages: list, source_filter: str | None = None) -> dict:
    body = {"messages": messages}
    if source_filter and source_filter != "all":
        body["knowledgeSourceParams"] = [
            {"kind": "searchIndex", "knowledgeSourceName": source_filter, "filterAddOn": ""}
        ]
    url = f"{SEARCH_ENDPOINT}/agents('{AGENT_NAME}')/retrieve?api-version={API_VERSION}"
    headers = {"Authorization": f"Bearer {_search_token}", "Content-Type": "application/json"}
    delay = 5
    for attempt in range(4):
        r = requests.post(url, headers=headers, json=body, timeout=120)
        if r.status_code == 429:
            print(f"   ⏳ Search 429 — {delay}s 대기 ({attempt+1}/4)")
            time.sleep(delay); delay *= 2
            continue
        if r.status_code >= 400:
            return {"error": r.text}
        return r.json()
    return {"error": "rate-limit-exhausted"}


def knowledge_agent_retrieve(query: str, source_filter: str = "all") -> str:
    """Foundry agent tool: KA 호출 후 텍스트 + reference 메타를 합쳐 반환"""
    resp = _retrieve(
        [{"role": "user", "content": [{"type": "text", "text": query}]}],
        source_filter=source_filter,
    )
    if "error" in resp:
        return json.dumps({"error": resp["error"][:500]}, ensure_ascii=False)
    text_parts = []
    for msg in resp.get("response", []):
        for c in msg.get("content", []):
            if c.get("type") == "text":
                text_parts.append(c["text"])
    refs = []
    for ref in (resp.get("references") or [])[:8]:
        sd = ref.get("sourceData", {}) or {}
        refs.append({
            "score": ref.get("rerankerScore"),
            "case": sd.get("caseNumber") or sd.get("docNumber"),
            "title": sd.get("caseName") or sd.get("title"),
            "summary": (sd.get("summary") or sd.get("reply") or "")[:300],
        })
    return json.dumps({"answer": "\n".join(text_parts), "references": refs}, ensure_ascii=False)


COMPLEX_QUESTION = "다리가 무너져서 사람들이 많이 죽었는데, 관련된 사람들이 다 같이 처벌받은 판례 있어? 그리고 그 판례에서 인용한 판례도 같이 찾아줘."

from azure.ai.agents import AgentsClient
from azure.ai.agents.models import FunctionTool, ToolSet, ListSortOrder

ag_client = AgentsClient(endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=cred)

functions = FunctionTool(functions={knowledge_agent_retrieve})
toolset = ToolSet()
toolset.add(functions)
ag_client.enable_auto_function_calls(toolset)

FOUNDRY_AGENT_NAME = "legal-foundry-agent"
existing = [a for a in ag_client.list_agents() if a.name == FOUNDRY_AGENT_NAME]
if existing:
    agent = existing[0]
    print(f"♻️  재사용: {agent.id}")
else:
    agent = ag_client.create_agent(
        model="gpt-4o",
        name=FOUNDRY_AGENT_NAME,
        instructions=(
            "너는 한국 법률 리서치 어시스턴트다. knowledge_agent_retrieve tool 을 자유롭게 호출해 "
            "자료를 모은 뒤, 충분하다고 판단되면 사건번호를 인용한 종합 답변을 한국어로 생성하라.\n"
            "검색 결과의 answer 나 references 에서 다른 사건번호가 인용되어 있으면, "
            "그 사건번호로 추가 검색해 인용 판례의 내용도 함께 조사하라."
        ),
        toolset=toolset,
    )
    print(f"✨ 생성: {agent.id}")

print(f"\n📨 자율 루프 실행 중...\n")
t0 = time.time()
run = ag_client.create_thread_and_process_run(
    agent_id=agent.id,
    thread={"messages": [{"role": "user", "content": COMPLEX_QUESTION}]},
    toolset=toolset,
)
elapsed = time.time() - t0
print(f"🏁 status={run.status}, elapsed={elapsed:.1f}s")
if run.status == "failed":
    print(f"❌ {run.last_error}")

print(f"\n📊 usage: prompt={run.usage.prompt_tokens}, completion={run.usage.completion_tokens}")

messages = list(ag_client.messages.list(thread_id=run.thread_id, order=ListSortOrder.ASCENDING))
print(f"\n💬 thread 총 메시지 {len(messages)}개")

steps = list(ag_client.run_steps.list(thread_id=run.thread_id, run_id=run.id))
tool_steps = [s for s in steps if s.type == "tool_calls"]
print(f"🔧 서버측 tool 호출 단계: {len(tool_steps)}개")
for i, s in enumerate(tool_steps, 1):
    if hasattr(s.step_details, "tool_calls"):
        for tc in s.step_details.tool_calls:
            if hasattr(tc, "function"):
                args_preview = (tc.function.arguments or "")[:120]
                print(f"  step {i}: {tc.function.name} args={args_preview}")

print("\n=== 최종 답변 (Foundry 자율 결과) ===")
for m in messages:
    if m.role == "assistant":
        for c in m.content:
            if hasattr(c, "text"):
                print(c.text.value)
                print("---")

## 8-7. 시나리오 F — 인용 판례 자동 추적 (Citation Chase)

### 🎯 목표
"메인 판례를 검색"한 뒤, **본문에 인용된 다른 사건번호**를 LLM 이 스스로 발견하고 그 인용 판례까지 후속 조회하는 진정한 멀티-홉 reasoning 데모.

### 🔍 데이터 분석으로 찾은 시드
실제 인덱스를 스캔한 결과 **성수대교 붕괴 사건 (97도1740, 대법원 1997.11.28)** 을 시드로 선정:

| 항목 | 값 |
|---|---|
| 메인 사건 | `97도1740` 업무상과실치사·치상·일반교통방해·자동차추락 |
| 본문 인용 사건번호 | `94도35`, `96도1231` (둘 다 인덱스에 존재 ✓) |
| 추적 가능 깊이 | 2-hop (메인 → 인용 판례) |

### 🧠 에이전트 동작 패턴
1. **Hop 1**: 사용자 질문을 `knowledge_agent_retrieve` 로 검색 → 답변/references 에 `94도35`, `96도1231` 등 사건번호 등장
2. **자율 판단**: instructions 에 따라 LLM 이 "이 사건번호들을 추가로 조회하면 답변 품질이 향상됨" 을 인식
3. **Hop 2**: 추출한 사건번호로 `knowledge_agent_retrieve` 재호출 → 인용 판례의 판시사항·요지 획득
4. **종합**: 메인 판례 + 인용 판례를 통합한 답변 생성

### ⚠️ 스키마 제약
현재 `prec-court-index` 에는 명시적 `referencedCases` 필드가 없으므로, 인용 정보는 `fullText` 본문에 자연어로만 존재합니다. 따라서 LLM 의 패턴 인식(`\d{2,4}[가-힣]{1,2}\d{2,6}`) 능력에 의존합니다 — 이것이 이 시나리오의 핵심 어려움입니다.


In [ ]:
"""8-7. 시나리오 F — Citation Chase (사전 지식 차단 + KB 자율 선택 + 중복 호출 차단)"""
import sys, time, re, json
_log_path = '/tmp/chase_out.txt'
_log_f = open(_log_path, 'w')
class _Tee:
    def __init__(self, *s): self.s = s
    def write(self, d):
        for x in self.s: x.write(d)
    def flush(self):
        for x in self.s: x.flush()
_orig_stdout = sys.stdout
sys.stdout = _Tee(_orig_stdout, _log_f)
try:
    from azure.ai.agents.models import FunctionTool, ToolSet, ListSortOrder

    CHASE_AGENT_NAME = "legal-foundry-agent-chase"
    CHASE_INSTRUCTIONS = (
        "너는 한국 법률 자료 분석 전문가다. knowledge_agent_retrieve tool 로 다음 4개 source 중 "
        "필요한 것을 **스스로 판단해 선택**해야 한다:\n"
        "  - ks-prec   : 법원 판례 (대법원·고등·지방)\n"
        "  - ks-const  : 헌법재판소 결정례\n"
        "  - ks-interp : 법제처 법령해석례\n"
        "  - ks-admin  : 행정심판 재결례\n"
        "  - all       : 4개 source 동시 검색 (어느 KB 가 적합한지 모를 때)\n\n"
        "★ 절대 규칙 ★\n"
        "0) **사전 학습 지식으로 사건번호를 추측해 query 에 쓰지 마라.** "
        "사건번호(예: '97도1740', '94도35' 같이 \\d{2,4}[가-힣]{1,2}\\d{2,6} 형태)는 "
        "오직 직전 tool 호출 결과의 answer 또는 references 본문에 **실제로 등장한 경우에만** 후속 query 로 사용할 수 있다.\n"
        "1) **첫 번째 호출**은 반드시 사용자가 준 자연어 설명(사실관계·법리)만을 query 로 써라. "
        "이 호출에는 절대 사건번호를 포함하지 마라. "
        "사용자 질문의 성격(형사·민사·헌법·행정·법령해석 등)을 분석해 가장 적합한 source_filter 를 골라라. "
        "확신이 없으면 source_filter='all' 로 시작해 결과를 보고 좁혀라.\n"
        "2) 첫 결과의 answer·references 를 자세히 읽어 메인 사건의 사건번호를 식별하고, "
        "그 본문에 **인용된 다른 사건번호**가 있는지 확인하라. "
        "사건번호를 하나도 찾지 못했다면 같은 query 를 반복하지 말고 **다른 표현/키워드로 재구성** 하거나 "
        "source_filter 를 바꿔서 호출하라.\n"
        "3) 발견된 인용 사건번호가 질문과 직접 관련 있어 보이면, "
        "**그 번호 자체를 query 로 한 후속 호출**을 수행하라 (예: query='94도35'). "
        "이때도 가장 적합한 source_filter 를 스스로 선택하라.\n"
        "4) 인덱스에 없거나 무관하면 그 사실을 답변에 명시하라.\n"
        "5) 무한 루프 방지: 추가 호출 최대 4회. "
        "**동일하거나 거의 동일한 query 문자열을 두 번 이상 호출하지 마라** "
        "(직전 호출과 의미가 같으면 즉시 중단하고 그때까지의 결과로 답변을 작성하라). "
        "동일 (source, 사건번호) 쌍도 재조회 금지.\n\n"
        "최종 답변에는 (a) 어느 KB 를 왜 선택했는지, (b) 메인 사례 요지 + 사건번호, "
        "(c) 추적한 인용 사건별 요지 + 사건번호, (d) 인용 관계가 본 사안에 미치는 의미를 한국어로 정리하라. "
        "모든 사건번호는 [사건번호] 형식으로 인용하라."
    )

    # --- Tool-side guard: 동일 (source, query) 재호출 차단 + 호출별 raw KA 응답 캡처 ---
    _seen_calls: set = set()
    _call_log: list = []   # [{order, source, query, blocked, sub_queries:[{ks,q,hits}], references:[{case,title}]}]
    def chase_retrieve(query: str, source_filter: str = "all") -> str:
        """[Tool] Citation-chase 전용 wrapper — 중복 호출 차단 + KA query plan 캡처."""
        key = (source_filter or "all", (query or "").strip())
        if key in _seen_calls:
            _call_log.append({"order": len(_call_log)+1, "source": key[0], "query": key[1],
                              "blocked": True, "sub_queries": [], "references": []})
            return json.dumps({
                "error": "duplicate_call_blocked",
                "message": f"이미 source={key[0]} 로 동일 query 를 호출했다. 다른 키워드/사건번호로 바꾸거나 지금까지 모은 자료로 최종 답변을 작성하라.",
            }, ensure_ascii=False)
        _seen_calls.add(key)
        # KA 직접 호출 (raw 응답에서 activity/references 추출 → log)
        raw = _retrieve(  # noqa: F821 — 셀 8-6 에서 정의됨
            [{"role": "user", "content": [{"type": "text", "text": query}]}],
            source_filter=source_filter,
        )
        sub_q = []
        if isinstance(raw, dict):
            for a in raw.get("activity", []) or []:
                if a.get("type") == "searchIndex":
                    args = a.get("searchIndexArguments", {}) or {}
                    sub_q.append({"ks": a.get("knowledgeSourceName"),
                                  "q": args.get("search", ""),
                                  "hits": a.get("count", 0)})
        refs = []
        for ref in ((raw.get("references") if isinstance(raw, dict) else None) or [])[:5]:
            sd = ref.get("sourceData", {}) or {}
            refs.append({"case": sd.get("caseNumber") or sd.get("docNumber"),
                         "title": sd.get("caseName") or sd.get("title")})
        _call_log.append({"order": len(_call_log)+1, "source": key[0], "query": key[1],
                          "blocked": False, "sub_queries": sub_q, "references": refs})
        if isinstance(raw, dict) and "error" in raw:
            return json.dumps({"error": str(raw["error"])[:500]}, ensure_ascii=False)
        text_parts = []
        for msg in raw.get("response", []):
            for c in msg.get("content", []):
                if c.get("type") == "text":
                    text_parts.append(c["text"])
        return json.dumps({
            "answer": "\n".join(text_parts),
            "references": refs,
        }, ensure_ascii=False)

    functions = FunctionTool(functions={chase_retrieve})
    chase_toolset = ToolSet()
    chase_toolset.add(functions)
    ag_client.enable_auto_function_calls(chase_toolset)  # noqa: F821

    # 동일 이름 agent 가 있으면 재사용 (instructions/toolset 변경 시 update_agent)
    existing_chase = [a for a in ag_client.list_agents() if a.name == CHASE_AGENT_NAME]  # noqa: F821
    if existing_chase:
        chase_agent = ag_client.update_agent(  # noqa: F821
            agent_id=existing_chase[0].id,
            instructions=CHASE_INSTRUCTIONS,
            toolset=chase_toolset,
        )
        print(f"♻️  재사용: {chase_agent.id}")
    else:
        chase_agent = ag_client.create_agent(  # noqa: F821
            model="gpt-4o", name=CHASE_AGENT_NAME,
            instructions=CHASE_INSTRUCTIONS, toolset=chase_toolset,
        )
        print(f"✨ 생성: {chase_agent.id}")

    CHASE_QUESTION = "다리가 무너져서 사람들이 많이 죽었는데, 관련된 사람들이 다 같이 처벌받은 판례 있어? 그리고 그 판례에서 인용한 판례도 같이 찾아줘."

    print(f"\n📨 Citation-chase 자율 루프 실행 중...\n")
    print(f"질문: {CHASE_QUESTION}\n")
    _seen_calls.clear(); _call_log.clear()
    t0 = time.time()
    run = ag_client.create_thread_and_process_run(  # noqa: F821
        agent_id=chase_agent.id,
        thread={"messages": [{"role": "user", "content": CHASE_QUESTION}]},
        toolset=chase_toolset,
    )
    elapsed = time.time() - t0
    print(f"🏁 status={run.status}, elapsed={elapsed:.1f}s")
    if run.status == "failed":
        print(f"❌ {run.last_error}")

    print(f"\n📊 usage: prompt_tokens={run.usage.prompt_tokens}, completion_tokens={run.usage.completion_tokens}")

    # ⚠️ run_steps.list 는 기본 DESCENDING — 시간순(ASC) 으로 재정렬
    steps_desc = list(ag_client.run_steps.list(thread_id=run.thread_id, run_id=run.id))  # noqa: F821
    steps = list(reversed(steps_desc))  # 오래된 → 최신
    tool_steps = [s for s in steps if s.type == "tool_calls"]
    print(f"\n🔧 서버측 tool 호출 단계: {len(tool_steps)}개  (시간순)")

    queries_issued: list = []
    sources_chosen: list = []
    for i, s in enumerate(tool_steps, 1):
        if hasattr(s.step_details, "tool_calls"):
            for tc in s.step_details.tool_calls:
                if hasattr(tc, "function"):
                    try:
                        args = json.loads(tc.function.arguments or "{}")
                        q = args.get("query", "")[:200]
                        src = args.get("source_filter", "all")
                    except Exception:
                        q = (tc.function.arguments or "")[:200]
                        src = "?"
                    queries_issued.append(q)
                    sources_chosen.append(src)
                    print(f"  step {i}: {tc.function.name}(source={src}, query={q!r})")

    # === Knowledge Agent 내부 query plan (각 retrieve 호출별 sub-query) ===
    print(f"\n🧩 Knowledge Agent 내부 query plan (호출 순서대로)")
    for entry in _call_log:
        tag = " ❌blocked" if entry["blocked"] else ""
        print(f"\n  ▶ retrieve #{entry['order']}  source={entry['source']}  query={entry['query']!r}{tag}")
        if entry["blocked"]:
            continue
        if not entry["sub_queries"]:
            print(f"      (sub-query 정보 없음)")
        for sq in entry["sub_queries"]:
            print(f"      • ks={(sq['ks'] or ''):<10} hits={sq['hits']:>3}  q={sq['q'][:80]!r}")
        if entry["references"]:
            print(f"      references(top {len(entry['references'])}):")
            for ref in entry["references"]:
                print(f"        - [{ref['case']}] {(ref['title'] or '')[:60]}")

    CITE_RE = re.compile(r"\d{2,4}[가-힣]{1,2}\d{2,6}")
    first_query = queries_issued[0] if queries_issued else ""
    followup_queries = queries_issued[1:]
    first_has_case_no = bool(CITE_RE.search(first_query))
    followup_case_nos: set = set()
    for q in followup_queries:
        followup_case_nos.update(CITE_RE.findall(q))

    thread_messages_full = list(ag_client.messages.list(thread_id=run.thread_id, order=ListSortOrder.ASCENDING))  # noqa: F821
    observed_case_nos: set = set()
    for m in thread_messages_full:
        for c in m.content:
            text = getattr(getattr(c, "text", None), "value", None) or ""
            observed_case_nos.update(CITE_RE.findall(text))

    grounded = followup_case_nos & observed_case_nos
    hallucinated = followup_case_nos - observed_case_nos
    duplicate_count = len(queries_issued) - len(set(zip(sources_chosen, queries_issued)))

    print(f"\n🔎 첫 호출 source 선택: {sources_chosen[0] if sources_chosen else '(없음)'}")
    print(f"🔎 첫 호출 query     : {first_query!r}")
    print(f"🔎 사용된 source 분포  : {dict((s, sources_chosen.count(s)) for s in sorted(set(sources_chosen)))}")
    print(f"🔎 중복 (source, query) 호출 차단 수: {duplicate_count}")
    print(f"🔎 첫 호출 query 에 사건번호 포함 여부: {'❌ 포함됨 (사전 지식 의심)' if first_has_case_no else '✅ 없음 (자연어 검색)'}")
    print(f"🔎 후속 호출에서 사용된 사건번호: {sorted(followup_case_nos) or '(없음)'}")
    print(f"   ├─ 검색 결과/응답에 실제 등장한 번호 (grounded): {sorted(grounded) or '(없음)'}")
    print(f"   └─ 검색 결과에 없었던 번호 (hallucinated 의심): {sorted(hallucinated) or '(없음)'}")

    if not first_has_case_no and grounded:
        print(f"\n✅ Grounded Citation Chase 성공: {len(grounded)}개 인용 판례를 검색 결과 기반으로 추적함")
    elif first_has_case_no:
        print(f"\n⚠️  첫 호출부터 사건번호 사용 — 모델이 사전 지식으로 prefetch 한 정황")
    elif hallucinated and not grounded:
        print(f"\n⚠️  후속 query 의 사건번호가 검색 결과에 등장하지 않음 — hallucination 위험")
    else:
        print(f"\n⚠️  후속 chase 가 발생하지 않음 — 메인 검색만으로 답변 종료")

    print("\n=== 최종 답변 (Citation Chase 결과) ===")
    for m in thread_messages_full:
        if m.role == "assistant":
            for c in m.content:
                if hasattr(c, "text"):
                    print(c.text.value)
                    print("---")
finally:
    sys.stdout = _orig_stdout
    _log_f.close()
print(f"(전체 출력 저장: {_log_path})")
